# 1-1. Prepare Training Data

In this notebook, we go through steps to :
- embed the population size measurement into the `AnnData.uns['pop']`.
- rename timepoint obs key
- perform dimension reduction : diffusion map 
- sample local state transition
- train test split

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
if sys.platform.startswith("darwin"):
    os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scanpy as sc
sc.settings.set_figure_params(frameon=False, dpi=30)

import our python package as `pdp` (***p***seudo ***d***ynamics ***p***lus)

In [ ]:
import pseudodynamics as pdp


In [ ]:
os.chdir(pdp.main_dir)
print("workding directory changed to:", pdp.main_dir)

## mouse *in vivo* BM haematopoiesis  

In this tutorial, we use a special persistent labling single-cell data from [Kucinski & Barile et. al, Cell Stem Cell. 2024](https://doi.org/10.1016/j.stem.2023.12.001).  

Persistent labeling technique can induces fluorescence in Hoxb5-expressing cells and propogate the label to their progenies. As a result, we can identify offsprings of haematopoitic stem and progenitor cells (HSPC) produced after induction from the by checking the tomato fluorescence (Tom+). 

  
This dataset contains 130700 cells from 9 timepoints spanning 9 month's time. At each time point we counted the total amount of Tom+ cells in the bone marrow of the mice.


The data is available at [figshare](https://figshare.com/ndownloader/articles/25398766/versions/3) 👈, or you can download it with the bash script `data_downloading.sh`.

In [ ]:
# we can download the expression matrix together with experimental settings

# create a data folder 
if not os.path.exists("data"):
    os.mkdir("data")

In [ ]:
adata_full = sc.read_h5ad("data/tompos/combined_filt.h5ad")
adata_full

### Data overview and filtering
The preprocess single-cell landscape is batch-correted and cell type annotated.

In [ ]:
sc.pl.umap(adata_full, 
           color=['timepoint_tx_days', 'tom', 'dpt_pseudotime',
                         'batch',  'anno_man', 'biosample_id', ], 
           ncols=3, wspace=0.3)

filtering Tom+ cells   

For this particular data, the newly generated cells are labeled with fluorensence. We thus filtered Tomato-positive (Tom+) cells and count the number of all the Tom+ cells in the bone marrow. At each time point, we sequenced and FACS-counted cells from several mouse. 

In [ ]:
# select positive cells and saved under data
adata = adata_full[adata_full.obs['tom'] == 'pos'].copy()
adata.write_h5ad('data/tom_pos.h5ad')

<font size=4 color='red'>Important !!!  </font> 


<font size=4> To train the model, you need to placed the processed data under `data`.
</font>

## 1. Attaching population size measurement

One of the main innovations of ***pseudodyanmics+*** is the use of population/tissue size measurement that comes from additional experiment, considering single-cell experiment has the issue of limiting capacity and sampling bias. 

However, for dataset that without tissue size measurement, we can go back to sequenced cell number in the sc library. <font color='red'> The parameters inferred in this manner may fail to reflect the absolute tissue growth and flux transiting between cell state. </font>

The population info is stored as a dictionary and saved in `adata.uns['pop']`. The `pop` dictionary must contain the following keys:
- `'t'` : the real experiment time point when the cells are counted and sequenced (assume on the same day)
- `'mean'` : the mean cell number of different experimental repeats for each time point
- `'std'` : the standard derivation of different experimental repeats 
- `'n_lib'` : the number of repeats 

For example, a single cell data with 3 time points (say day0, day4, day10) and 3 repeats each time looks like this:
 
```Python
    adata.uns['pop'] = {
        "t" = np.array([0, 4, 10]),
        "mean" = np.array([mean_d0, mean_d4, mean_d10]),
        "std" = np.array([std_d0, std_d4, std_d10]),
        "n_lib" = np.array([3, 3, 3])
    }
```

In [ ]:
adata = sc.read_h5ad('data/tom_pos.h5ad')
adata

### load FACS data

Here we are loading the FACS experiment data. The BM cells of the same mouse are sorted for tomato positive and count total cell number before sending for sequencing.

In [ ]:
pop_df = pd.read_csv("data/tompos/model_input_tompos.csv")
pop_df.head()

We summarize the total cell number over different mouse for each time point

In [ ]:
pop_by_time = pop_df.groupby("time").agg({"flow_total":["mean", "std","count"], "sc_ncells_filt":["mean", "std"]})
pop_by_time

In [ ]:
import matplotlib.pyplot as plt


def plot_mean_std(col_name, ax):
    time_points = pop_by_time.index
    mean_values = pop_by_time[(col_name, 'mean')]
    std_values = pop_by_time[(col_name, 'std')]

    ax.plot(range(len(time_points)), mean_values, marker='o', label=col_name)
    ax.fill_between(range(len(time_points)), 
                    mean_values - std_values, 
                    mean_values + std_values, 
                    alpha=0.3, color='gray', label=col_name+"_error")
    ax.set_xticks(range(len(time_points)))
    ax.set_xticklabels(time_points)

fig = plt.figure(dpi=80, figsize=(6, 3))
ax = fig.gca()
plot_mean_std('flow_total', ax=ax)

Here we found that the measurment at Day 161 seems to be an outlier. We do the following correction :

- we estimate the average growth rate from Day112 to Day269 with $ N_t = N_0 * e^{g \Delta t}$
- next we impute the mean pop size at Day161 with the average growth rate

In [ ]:
time_points = pop_by_time.index
mean_values = pop_by_time[('flow_total', 'mean')].values
g = np.log(mean_values[-1]  / mean_values[-3]) / (time_points[-1] - time_points[-3]) 
print(f'the estimated mean g is {g:.5f}')

impute_Day161 = np.exp(g*(time_points[-2] - time_points[-3]) ) * mean_values[-3]
print(f'the imputed mean pop size is {impute_Day161:.2f}')

# replace the mean value
mean_values[-2] = impute_Day161

In [ ]:
fig = plt.figure(dpi=80, figsize=(6, 3))
ax = fig.gca()
plot_mean_std('flow_total', ax)
plot_mean_std('sc_ncells_filt', ax)
ax.legend()

In [ ]:
# assign to uns['pop']

adata.uns['pop'] = {
    't' : time_points.to_numpy(),
    'mean' : mean_values,
    'std' : pop_by_time[('flow_total', 'std')].values,
    'n_lib' : pop_by_time[('flow_total', 'count')].values
}

adata.uns['pop'] 

## 2. rename Timepoint Key 

We use a unique obs key `timepoint_tx_days` to store the timepoint.

In [ ]:
if 'timepoint_tx_days' not in adata.obs_keys():
    adata.obs['timepoint_tx_days'] = adata.obs['day'].astype(int) # example

check if the time point from the pop info match with that in the sequencing data

In [ ]:
adata.obs['timepoint_tx_days'].isin(adata.uns['pop']['t']).all()

## 3. Compute diffusion map (DM)

The example dataset has pre-computed diffusion maps. For reproducibility, plesae keep the original diffusion map and pseudotime.  
Here, for demonstration, we show two popular methods of getting DM cooridnates and pseudotime.  
- scanpy diffusion map and dpt_psuedotime 
- palantir diffusion map and palantir pseudotime

In [ ]:
adata

In [ ]:
if 'X_diffmap' not in adata.obsm_keys():
    # use batch corrected cell representation like Harmonied PC or SCVI latent
    sc.pp.neighbors(adata, n_pcs=30, use_rep='X_pca_harmony') 
    sc.tl.diffmap(adata, n_comps=10)

adata.uns['iroot']= np.argmin(adata.obs['HSCscore'].values)
sc.tl.dpt(adata)

sc.pl.umap(adata, color=['HSCscore', 'dpt_pseudotime'])

In [ ]:
# next we scaled the diffusion map to 0-1.
DM = adata.obsm['X_diffmap']
DM_min = DM.min(axis=0,keepdims=True)
DM_range = DM.max(axis=0,keepdims=True) - DM_min
DM_scaled = (DM - DM_min)/DM_range
adata.obsm['DM_scaled'] = DM_scaled
print(f"shape of DM : {DM.shape} and scaled DM : {DM_scaled.shape} ")

Use palantir to compute DM 

In [ ]:
import palantir
dm_res = palantir.utils.run_diffusion_maps(adata, pca_key='X_pca_harmony')
ms_data = palantir.utils.determine_multiscale_space(adata)

print(f"shape of palantir DM : {adata.obsm['DM_EigenVectors'].shape} and scaled DM : {adata.obsm['DM_EigenVectors_multiscaled'].shape} ")

In [ ]:
adata

## 4. Sample local cell state changes in diffusion map

***Pseudodynamics+*** can make use of the local cell state changes in diffusion map to assists learning velocity, inspired by [TrajectoryNet by Tong et. al.](https://arxiv.org/abs/2002.04461). This state changes is defined as $\Delta x = \sum_{i \in N(x)} \frac{w_{i,x}}{w_{i,x} + \sum_{j \in N(x)} w_{i,j}} \Delta x_i$ ,Here we provdie two ways of sampling this local state transition:

- by default we rely on KNN and pseudotime with $w_{i,j}$ denoting the connectivities in KNN.

- alternatively, when a transition matrix is provided, e.g. cellrank. The $w_{i,j}$ denote the probability of transitioning from $i$ to $j$.

In [ ]:
delta_DM , neighbors = pdp.tl.sample_deltax(adata, xkey='DM_scaled', pseudotimekey='dpt_pseudotime', repeat=15)
delta_DM_ay = np.stack(delta_DM)

print(delta_DM_ay.shape)
adata.obsm['Delta_DM'] = delta_DM_ay.mean(axis=-1) # use the averaged ∆DM

## 5. train test split

In [ ]:
def train_test_split_adata(adata,  leaveout=None, val_size=0.1, test_size=0.1, timepoint_key='timepoint_tx_days'):

    obs = adata.obs
    val_test_cbs = obs.query(f"`{timepoint_key}` not in @leaveout").sample(frac=0.2).index
    test_cb = np.random.choice(val_test_cbs, size=len(val_test_cbs)//2,replace=False)

    split_mapper = {cb:'test' for cb in test_cb}
    val_cbs = {cb:'val' for cb in val_test_cbs if cb not in test_cb}
    train_cbs = {cb:'train' for cb in adata.obs_names if cb not in val_test_cbs}

    split_mapper.update(val_cbs)
    split_mapper.update(train_cbs)

    return adata.obs.index.map(split_mapper)

adata.obs['split'] = train_test_split_adata(adata, leaveout=[3])

In [ ]:
adata.write_h5ad("data/tom_pos.h5ad")

In [ ]:
adata.obsm['DM_scaled']

**What's next**
- Estimate single-cell density at observed time point (optional). 
- set up training configuration
